# Revision controls — independent S–V–T split, transfer-richer families, fixed term count, depolarizing noise, expressibility

Companion notebook to the revised manuscript *Auditing Trajectory-Based Transfer Diagnostics in Variational Quantum Learning* (Electronics, electronics-4514256, revision R1). It bundles the runner used for the new controls (same Reptile meta-initialization, $K=10$, learning rates and ansatz as the main sweep; `scaling_sweep.py` from the public repository must be importable from the working directory), the analysis that produces the pooled and per-seed tables of Sec. 3.6 and Supplementary Secs. S7–S10, and the expressibility estimator.

Set `RUN_EXPERIMENTS = True` to regenerate the per-task CSVs in `rev_results/` (roughly one to two hours on a single CPU core; the noise runs use a density-matrix backend). With the shipped CSVs in place the analysis cells reproduce the tables directly.

In [1]:
RUN_EXPERIMENTS = False
RUN_EXPRESSIBILITY = False
PLAN_FILES = ['plan.json', 'plan2.json', 'plan3.json']   # run configurations (skipped when the CSV already exists)

In [2]:
import os, json, time, numpy as np, pandas as pd
from scipy import stats
import pennylane as qml
from pennylane import numpy as pnp
import scaling_sweep as ss
print('pennylane', qml.__version__)

pennylane 0.45.1


## Runner (families, splits, fixed term count, noise)

In [3]:
"""Revision experiments: independent S-V-T split, transfer-rich family, fixed-term-count, depolarizing noise.
Reuses the paper's pipeline functions (scaling_sweep.py): Reptile meta-init (200 outer, K=10, lr 0.5/0.3, init 0.1)."""
import sys, os, json, time, numpy as np, pandas as pd, pennylane as qml
from pennylane import numpy as pnp
from scipy import stats
import scaling_sweep as ss

def draw_coeffs_corr(rng, n, lo=0.5, hi=1.5, aniso=0.15, field_scale=0.5, site_noise=0.15):
    """Transfer-rich family: XX/YY/ZZ on a bond share one coupling (small anisotropy); fields share a common value."""
    M, nb = 4*n-3, n-1
    J = rng.uniform(lo, hi, size=nb) * rng.choice([-1.0, 1.0], size=nb)
    c = np.empty(M)
    for b in range(nb):
        for a in range(3):
            c[3*b+a] = J[b] * (1.0 + aniso*rng.normal())
    h = rng.normal(0.0, field_scale)
    c[3*nb:] = h + site_noise*rng.normal(size=n)
    return c


def draw_coeffs_unif(rng, n, lo=0.5, hi=1.5, aniso=0.05, bond_noise=0.05, field_scale=0.5, site_noise=0.05):
    """Strong-transfer family: one coupling J for all bonds and all XX/YY/ZZ (5% noise), one common field h (5% site noise)."""
    M, nb = 4*n-3, n-1
    J = rng.uniform(lo, hi) * rng.choice([-1.0, 1.0])
    c = np.empty(M)
    for b in range(nb):
        Jb = J * (1.0 + bond_noise*rng.normal())
        for a in range(3):
            c[3*b+a] = Jb * (1.0 + aniso*rng.normal())
    h = rng.normal(0.0, field_scale)
    c[3*nb:] = h * (1.0 + site_noise*rng.normal(size=n))
    return c

def split3(rng, M, fr=(0.5, 0.25, 0.25)):
    perm = rng.permutation(M); s = int(round(fr[0]*M)); v = int(round(fr[1]*M))
    return perm[:s], perm[s:s+v], perm[s+v:]

def make_loss(dev, n, L, coeffs, ops, p):
    denom = float(np.sum(np.abs(coeffs))) or 1.0
    H = qml.Hamiltonian([float(x)/denom for x in coeffs], list(ops))
    @qml.qnode(dev, diff_method="backprop")
    def qn(theta):
        for l in range(L):
            for w in range(n):
                qml.RY(theta[l, w, 0], wires=w); qml.RZ(theta[l, w, 1], wires=w)
            for w in range(n-1): qml.CNOT(wires=[w, w+1])
            if p > 0:
                for w in range(n): qml.DepolarizingChannel(p, wires=w)
        return qml.expval(H)
    return qn

def grad(qn, th): return np.asarray(qml.grad(qn)(th), dtype=float).reshape(-1)

def task_draw(rng, n, ops, family, split, fixed_M):
    c = {"orig": ss.draw_coeffs, "corr": draw_coeffs_corr, "unif": draw_coeffs_unif}[family](rng, n)
    idx = np.arange(len(ops))
    if fixed_M: idx = rng.choice(len(ops), fixed_M, replace=False)
    c_act, ops_act = c[idx], [ops[j] for j in idx]
    if split == "2way":
        S, V = ss.split_indices(rng, len(idx)); T = None
    else:
        S, V, T = split3(rng, len(idx))
    return c_act, ops_act, S, V, T

def run_cfg(cfg):
    n, L, seed = cfg["n"], cfg["L"], cfg["seed"]; family, split = cfg["family"], cfg["split"]
    fixed_M, p = cfg.get("fixed_M"), cfg.get("noise", 0.0)
    K, inner_lr, outer_lr, n_outer, n_eval, init_scale = 10, 0.5, 0.3, cfg.get("n_outer", 200), cfg.get("n_eval", 50), 0.1
    dev = qml.device("default.mixed" if p > 0 else "default.qubit", wires=n)
    ops = ss.build_ops(n)
    rng = np.random.default_rng(cfg.get("base", 0) + 1000*n + 100*L + seed)
    # Reptile meta-initialization on support losses of the family
    theta = pnp.array(rng.normal(0.0, init_scale, size=(L, n, 2)), requires_grad=True)
    for _ in range(n_outer):
        c, oa, S, V, T = task_draw(rng, n, ops, family, split, fixed_M)
        qS = make_loss(dev, n, L, c[S], [oa[j] for j in S], p)
        phi = pnp.array(theta, requires_grad=True)
        for _ in range(K):
            phi = pnp.array(phi - inner_lr*qml.grad(qS)(phi), requires_grad=True)
        theta = pnp.array(theta + outer_lr*(phi - theta), requires_grad=True)
    rows = []
    for t in range(n_eval):
        c, oa, S, V, T = task_draw(rng, n, ops, family, split, fixed_M)
        qS = make_loss(dev, n, L, c[S], [oa[j] for j in S], p)
        qV = make_loss(dev, n, L, c[V], [oa[j] for j in V], p)
        qT = make_loss(dev, n, L, c[T], [oa[j] for j in T], p) if T is not None else None
        gS, gV = grad(qS, theta), grad(qV, theta)
        r = dict(n=n, L=L, seed=seed, task=t, family=family, split=split, fixed_M=fixed_M or 0, noise=p,
                 norm_gS=float(np.linalg.norm(gS)), norm_gV=float(np.linalg.norm(gV)),
                 A_V=float(gS@gV/(np.linalg.norm(gS)*np.linalg.norm(gV))),
                 LS0=float(qS(theta)), LV0=float(qV(theta)))
        if qT is not None:
            gT = grad(qT, theta); r["A_T"] = float(gS@gT/(np.linalg.norm(gS)*np.linalg.norm(gT))); r["LT0"] = float(qT(theta))
        phi = pnp.array(theta, requires_grad=True)
        for _ in range(K):
            phi = pnp.array(phi - inner_lr*qml.grad(qS)(phi), requires_grad=True)
        r["LSK"], r["LVK"] = float(qS(phi)), float(qV(phi))
        r["I_S"], r["I_V"] = r["LS0"]-r["LSK"], r["LV0"]-r["LVK"]; r["G_V"] = r["LVK"]-r["LSK"]
        if qT is not None:
            r["LTK"] = float(qT(phi)); r["I_T"] = r["LT0"]-r["LTK"]; r["G_T"] = r["LTK"]-r["LSK"]
        r["drift"] = float(np.linalg.norm(np.asarray(phi, dtype=float).reshape(-1) - np.asarray(theta, dtype=float).reshape(-1)))
        rows.append(r)
    df = pd.DataFrame(rows)
    # barren-plateau probe at random init on the same (possibly fixed-M) support loss
    if cfg.get("bp_probe"):
        comp, norms = [], []
        for _ in range(cfg["bp_probe"]):
            c, oa, S, V, T = task_draw(rng, n, ops, family, split, fixed_M)
            qS = make_loss(dev, n, L, c[S], [oa[j] for j in S], p)
            th = pnp.array(rng.uniform(0, 2*np.pi, size=(L, n, 2)), requires_grad=True)
            g = grad(qS, th); comp.append(g[0]); norms.append(np.linalg.norm(g))
        df["bp_var"] = float(np.var(comp)); df["bp_norm_mean"] = float(np.mean(norms))
    return df



## Run configurations

In [4]:
if RUN_EXPERIMENTS:
    os.makedirs('rev_results', exist_ok=True); seen = set()
    for pf in PLAN_FILES:
        if not os.path.exists(pf): continue
        for cfg in json.load(open(pf)):
            tag = f"{cfg['family']}_{cfg['split']}_n{cfg['n']}_L{cfg['L']}_M{cfg.get('fixed_M') or 0}_p{cfg.get('noise',0)}_s{cfg['seed']}"
            out = f'rev_results/{tag}.csv'
            if out in seen or os.path.exists(out): continue
            seen.add(out); t0 = time.time(); run_cfg(cfg).to_csv(out, index=False); print(tag, f'{time.time()-t0:.0f}s')
else:
    print('RUN_EXPERIMENTS=False: using shipped rev_results/*.csv')

RUN_EXPERIMENTS=False: using shipped rev_results/*.csv


## Analysis — pooled and per-seed tables (Sec. 3.6, Table 5; Supplementary Tables SVIII–SXI)

In [5]:
import glob, os, numpy as np, pandas as pd
from scipy import stats
def rho(x, y): return stats.spearmanr(x, y)[0]
def partial(x, y, Z):
    xr, yr = stats.rankdata(x), stats.rankdata(y)
    Zr = np.column_stack([stats.rankdata(z) for z in Z] + [np.ones(len(xr))])
    rx = xr - Zr @ np.linalg.lstsq(Zr, xr, rcond=None)[0]; ry = yr - Zr @ np.linalg.lstsq(Zr, yr, rcond=None)[0]
    return stats.pearsonr(rx, ry)[0]
def load():
    fs = sorted(glob.glob("rev_results/*.csv")); df = pd.concat([pd.read_csv(f) for f in fs], ignore_index=True) if fs else pd.DataFrame()
    return df
def cfg_key(d): return (d.family.iloc[0], d.split.iloc[0], int(d.n.iloc[0]), int(d.L.iloc[0]), int(d.fixed_M.iloc[0]), float(d.noise.iloc[0]))
def stats_block(d):
    out = dict(n_tasks=len(d), seeds=d.seed.nunique(), mean_IV=d.I_V.mean(), sd_IV=d.I_V.std(), se_IV=d.I_V.std()/np.sqrt(len(d)),
               rho_AV_IV=rho(d.A_V, d.I_V), raw_AV_GV=rho(d.A_V, d.G_V), AV_GV_given_IV=partial(d.A_V, d.G_V, [d.I_V]),
               AV_GV_joint=partial(d.A_V, d.G_V, [d.I_V, d.LV0]), cv_gS=d.norm_gS.std()/d.norm_gS.mean(), mean_norm_gS=d.norm_gS.mean(), mean_drift=d.drift.mean())
    if "I_T" in d and d.I_T.notna().all():
        out.update(mean_IT=d.I_T.mean(), se_IT=d.I_T.std()/np.sqrt(len(d)), rho_AV_IT=rho(d.A_V, d.I_T), rho_AT_IT=rho(d.A_T, d.I_T), rho_AV_AT=rho(d.A_V, d.A_T),
                   raw_AV_GT=rho(d.A_V, d.G_T), AV_GT_given_IV=partial(d.A_V, d.G_T, [d.I_V]), AV_GT_joint_IV_LT0=partial(d.A_V, d.G_T, [d.I_V, d.LT0]),
                   AV_GT_joint_IT_LT0=partial(d.A_V, d.G_T, [d.I_T, d.LT0]), rho_IV_IT=rho(d.I_V, d.I_T))
    if "bp_var" in d: out.update(bp_var=d.bp_var.mean(), bp_norm_mean=d.bp_norm_mean.mean())
    return out


In [6]:
df = load(); assert not df.empty, 'no rev_results/*.csv found'
rows, per_seed = [], []
for key, d in df.groupby(['family','split','n','L','fixed_M','noise']):
    s = stats_block(d); s.update(dict(zip(['family','split','n','L','fixed_M','noise'], key))); rows.append(s)
    for sd, ds in d.groupby('seed'):
        q = stats_block(ds); q.update(dict(zip(['family','split','n','L','fixed_M','noise'], key))); q['seed'] = sd; per_seed.append(q)
pooled, seeds = pd.DataFrame(rows), pd.DataFrame(per_seed)
os.makedirs('rev_tables', exist_ok=True); pooled.to_csv('rev_tables/pooled.csv', index=False); seeds.to_csv('rev_tables/per_seed.csv', index=False)
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 40)
cols = [c for c in ['family','split','n','fixed_M','noise','seeds','mean_IV','se_IV','mean_IT','rho_AV_IV','rho_AV_IT','rho_AT_IT','raw_AV_GV','AV_GV_joint','raw_AV_GT','AV_GT_joint_IT_LT0','cv_gS','mean_norm_gS','bp_var'] if c in pooled]
print(pooled[cols].round(3).to_string(index=False))

family split  n  fixed_M  noise  seeds  mean_IV  se_IV  mean_IT  rho_AV_IV  rho_AV_IT  rho_AT_IT  raw_AV_GV  AV_GV_joint  raw_AV_GT  AV_GT_joint_IT_LT0  cv_gS  mean_norm_gS  bp_var
  corr  2way  6        0   0.00      3    0.022  0.014      NaN      0.680        NaN        NaN     -0.343       -0.117        NaN                 NaN  0.239         0.309     NaN
  orig  2way  4       13   0.00      2   -0.006  0.028      NaN      0.527        NaN        NaN     -0.464       -0.102        NaN                 NaN  0.302         0.391   0.005
  orig  2way  6        0   0.01      2    0.012  0.011      NaN      0.728        NaN        NaN     -0.317        0.006        NaN                 NaN  0.256         0.253     NaN
  orig  2way  6       13   0.00      2   -0.026  0.022      NaN      0.648        NaN        NaN     -0.408       -0.267        NaN                 NaN  0.240         0.359   0.002
  orig  2way  8       13   0.00      1   -0.029  0.023      NaN      0.664        NaN        Na

## Expressibility (Sim et al. KL measure, 75 bins) — Supplementary Table SXII

In [7]:
def run_expressibility(n_pairs=1000, out_csv='rev_results_expressibility.csv'):
    rows = []; t0 = time.time()
    for n in [4, 6, 8, 10]:
        dev = qml.device("default.qubit", wires=n); N = 2**n
        for L in [2, 3, 5]:
            @qml.qnode(dev)
            def st(theta):
                ss.ansatz(theta, n, L); return qml.state()
            rng = np.random.default_rng(1234 + n*10 + L)
            F = np.array([abs(np.vdot(st(rng.uniform(0, 2*np.pi, size=(L, n, 2))), st(rng.uniform(0, 2*np.pi, size=(L, n, 2)))))**2 for _ in range(1000)])
            bins = np.linspace(0, 1, 76); p_pqc = np.histogram(F, bins=bins)[0]/len(F)
            p_haar = np.diff([1-(1-b)**(N-1) for b in bins]); m = p_pqc > 0
            kl = float(np.sum(p_pqc[m]*np.log(p_pqc[m]/p_haar[m])))
            rows.append(dict(n=n, L=L, d=2*n*L, expr_KL=round(kl, 3), meanF=round(float(F.mean()), 4), haar_meanF=round(1/N, 4)))
            print(n, L, round(kl,3), flush=True)
    ex = pd.DataFrame(rows)
    summ = pd.read_csv("repo/data/results_summary.csv").groupby(["n","L"]).agg(cv=("cv_gtr","mean"), mnorm=("mean_norm_gtr","mean"), ident=("rho_A_Ival","mean")).reset_index()
    mg = ex.merge(summ, on=["n","L"]); mg.to_csv("rev_results_expressibility.csv", index=False)
    for col in ["cv","mnorm","ident"]: print(f"Spearman(expr_KL,{col}) = {stats.spearmanr(mg.expr_KL, mg[col])[0]:+.3f}", flush=True)
    print("EXPR DONE", f"{time.time()-t0:.0f}s", flush=True)
    return mg

mg = run_expressibility() if RUN_EXPRESSIBILITY else pd.read_csv('rev_results_expressibility.csv')
print(mg[['n','L','expr_KL','meanF','haar_meanF','cv','mnorm','ident']].round(3).to_string(index=False))
for col in ['cv','mnorm','ident']: print(f'Spearman(expr_KL,{col}) = {stats.spearmanr(mg.expr_KL, mg[col])[0]:+.3f}')

 n  L  expr_KL  meanF  haar_meanF    cv  mnorm  ident
 4  2    0.018  0.065       0.062 0.326  0.337  0.762
 4  3    0.019  0.063       0.062 0.262  0.398  0.616
 4  5    0.010  0.061       0.062 0.298  0.467  0.467
 6  2    0.065  0.016       0.016 0.273  0.242  0.764
 6  3    0.014  0.016       0.016 0.233  0.285  0.665
 6  5    0.006  0.016       0.016 0.242  0.337  0.569
 8  2    0.032  0.004       0.004 0.196  0.207  0.805
 8  3    0.020  0.004       0.004 0.219  0.236  0.824
 8  5    0.001  0.004       0.004 0.186  0.285  0.626
10  2    0.013  0.001       0.001 0.195  0.151  0.920
10  3    0.006  0.001       0.001 0.160  0.178  0.842
10  5    0.000  0.001       0.001 0.187  0.221  0.811
Spearman(expr_KL,cv) = +0.501
Spearman(expr_KL,mnorm) = +0.039
Spearman(expr_KL,ident) = +0.102
